# Sicilian NMT with NLLB-200 — zero-shot + LoRA fine-tune

All-in-one: evaluate Meta's **NLLB-200** zero-shot on our Arba-Sicula test set, then **LoRA fine-tune** it on our training data and re-evaluate. Same BLEU/chrF as the local Sockeye baseline. Sicilian = `scn_Latn`.

**Steps:** Runtime → Change runtime type → **GPU** (T4 is fine). Run cells top to bottom; upload the six `data/dataset/` files (`train/valid/test.{scn,en}`) when prompted.

Reference on this test set (scn→en): floor (copy) BLEU 5.27 · Sockeye baseline ≈5.2 · Sockeye lever-B 7.24 (tokenized).

In [ ]:
!pip -q install transformers datasets peft sentencepiece sacrebleu accelerate

In [ ]:
# Upload train.scn train.en valid.scn valid.en test.scn test.en  (from data/dataset/)
from google.colab import files
_ = files.upload()
def read(p): return open(p, encoding='utf-8').read().splitlines()
splits = {s: {l: read(f'{s}.{l}') for l in ('scn', 'en')} for s in ('train', 'valid', 'test')}
print({s: len(d['scn']) for s, d in splits.items()})

In [ ]:
import torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from sacrebleu.metrics import BLEU, CHRF

MODEL = 'facebook/nllb-200-distilled-600M'   # or facebook/nllb-200-1.3B
LANG = {'scn': 'scn_Latn', 'en': 'eng_Latn'}
device = 'cuda' if torch.cuda.is_available() else 'cpu'
tok = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL).to(device).eval()

def translate(texts, src, tgt, bs=32, max_len=160):
    tok.src_lang = LANG[src]
    tgt_id = tok.convert_tokens_to_ids(LANG[tgt])
    out = []
    for i in range(0, len(texts), bs):
        enc = tok(texts[i:i+bs], return_tensors='pt', padding=True, truncation=True, max_length=max_len).to(device)
        with torch.no_grad():
            gen = model.generate(**enc, forced_bos_token_id=tgt_id, max_length=max_len, num_beams=5)
        out += tok.batch_decode(gen, skip_special_tokens=True)
    return out

def report(hyp, ref, tag):
    print(f'{tag}:  BLEU={BLEU().corpus_score(hyp,[ref]).score:.2f}  chrF={CHRF().corpus_score(hyp,[ref]).score:.2f}')
print('loaded', MODEL, 'on', device)

In [ ]:
# ---- ZERO-SHOT ----
report(translate(splits['test']['scn'], 'scn', 'en'), splits['test']['en'],  'zero-shot scn->en')
report(translate(splits['test']['en'],  'en', 'scn'), splits['test']['scn'], 'zero-shot en->scn')

## LoRA fine-tune (scn→en)

Parameter-efficient: base weights frozen, small adapters trained on our data. Change `DIRECTION` to `en2scn` to fine-tune the other way.

In [ ]:
from datasets import Dataset
from peft import LoraConfig, get_peft_model
from transformers import DataCollatorForSeq2Seq, Seq2SeqTrainer, Seq2SeqTrainingArguments

DIRECTION, EPOCHS = 'scn2en', 3
src, tgt = DIRECTION.split('2')
tok.src_lang, tok.tgt_lang = LANG[src], LANG[tgt]

ft = get_peft_model(model, LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, bias='none',
                                      target_modules=['q_proj', 'v_proj'], task_type='SEQ_2_SEQ_LM'))
ft.print_trainable_parameters()

def tokenize(b):
    return tok(b['src'], text_target=b['tgt'], max_length=160, truncation=True)
train_ds = Dataset.from_dict({'src': splits['train'][src], 'tgt': splits['train'][tgt]}).map(
    tokenize, batched=True, remove_columns=['src', 'tgt'])

args = Seq2SeqTrainingArguments(output_dir=f'nllb-lora-{DIRECTION}', num_train_epochs=EPOCHS,
    per_device_train_batch_size=16, learning_rate=2e-4, fp16=(device=='cuda'),
    logging_steps=50, save_strategy='no', report_to=[])
Seq2SeqTrainer(model=ft, args=args, train_dataset=train_ds,
               data_collator=DataCollatorForSeq2Seq(tok, model=ft)).train()

In [ ]:
# ---- EVALUATE FINE-TUNED ----  (model is now the LoRA-adapted ft)
model = ft.eval()
report(translate(splits['test'][src], src, tgt), splits['test'][tgt], f'LoRA fine-tuned {DIRECTION}')
ft.save_pretrained(f'nllb-lora-{DIRECTION}')
print('saved adapter to', f'nllb-lora-{DIRECTION}')

**Note:** NLLB's seed Sicilian uses a post-2017 orthography differing from the Arba-Sicula literary standard of our test set, so zero-shot may be penalised on chrF/BLEU; fine-tuning on our `train` is exactly the fix. NLLB numbers here are raw text (compare to the raw floor 5.27); the Sockeye 7.24 was tokenized space.